# Демо 3. SFT: від повного донавчання до LoRA і QLoRA

У цьому ноутбуці донавчаємо моделі **Qwen3** трьох розмірів виконувати одну вузьку прикладну задачу:
**класифікувати звернення клієнтів служби підтримки** й відповідати лише JSON із категорією та наміром звернення.

### Навіщо потрібен SFT

Інтернет-магазин щодня отримує тисячі повідомлень від клієнтів: скасувати замовлення, змінити адресу доставки,
повернути кошти тощо. Щоб автоматично передавати кожне звернення у відповідний відділ, програмі потрібна відповідь
у строго заданому форматі, наприклад `{"category": "ORDER", "intent": "cancel_order"}`, з мітками з переліку компанії.

Загальна instruct-модель може виконати таку задачу, якщо описати її в промпті, але:
- вона не знає переліку міток компанії, тому його доводиться передавати в кожному запиті;
- навіть із переліком вона часто плутає близькі мітки, бо не знає, як саме компанія їх розрізняє
  (наприклад, `get_refund` і `track_refund`).

Донавчання на прикладах із правильними мітками закріплює формат відповіді й значення кожної мітки у вагах моделі,
тому після нього достатньо короткого промпту.

План:
1. Підготувати дані у форматі `messages` і розбити їх на тренувальний, валідаційний і тестовий набори.
2. Підготувати функції для генерації й оцінювання.
3. Оцінити, чому повне донавчання великої моделі не вміщується в GPU T4.
4. Налаштувати **LoRA** і параметри тренування.
5. Донавчити методом LoRA моделі **Qwen3-0.6B, Qwen3-1.7B і Qwen3-4B** і порівняти, чи може менша модель після донавчання наздогнати більшу.
6. Виконати **повне донавчання** найменшої моделі й порівняти його з LoRA.
7. Зберегти адаптер, завантажити його знову й влити у ваги моделі.
8. Донавчити Qwen3-4B методом **QLoRA** і порівняти витрати памʼяті.

Приклад за мотивами розділу 11.4 [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter11/4)
і вправ курсу [smol-course](https://huggingface.co/learn/smol-course).
Для запуску в Colab оберіть середовище з GPU.

In [ ]:
# trl — бібліотека для донавчання мовних моделей (SFT, DPO тощо),
# peft — реалізація LoRA та інших методів PEFT, bitsandbytes — 4-бітна квантизація для QLoRA
!pip install -q "transformers>=4.55.0" "trl>=1.17" "peft" "bitsandbytes" "datasets" "accelerate"
# Colab має застарілу бібліотеку torchao, з якою peft не імпортується; для цього демо вона не потрібна
!pip uninstall -q -y torchao

In [1]:
import gc
import json
import os
import time

import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

assert torch.cuda.is_available(), "Для цього ноутбука потрібна GPU"
print(f"GPU: {torch.cuda.get_device_name()}")
print(f"Памʼять GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: NVIDIA GeForce GTX 1070
Памʼять GPU: 8.6 GB


## 1. Дані

[Bitext Customer Support](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset) —
датасет із майже 27 тис. повідомлень клієнтів англійською мовою. Кожне повідомлення має дві мітки:
- `category` — відділ, якого стосується звернення (11 категорій, наприклад `ORDER`, `REFUND`, `ACCOUNT`);
- `intent` — конкретний намір клієнта (27 намірів, наприклад `cancel_order`, `track_refund`).

Конкретні дані в повідомленнях замінено заповнювачами на кшталт `{{Order Number}}`.

In [2]:
raw_dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train")
print(raw_dataset)

for ex in raw_dataset.shuffle(seed=0).select(range(5)):
    print(f"{ex['category']:<10} {ex['intent']:<25} {ex['instruction']}")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

Generating train split: 100%|██████████| 26872/26872 [00:00<00:00, 368379.32 examples/s]

Dataset({
    features: ['flags', 'instruction', 'category', 'intent', 'response'],
    num_rows: 26872
})
SHIPPING   set_up_shipping_address   there are errors setting up a delivery address
PAYMENT    check_payment_methods     i want assistance seeing ur accepted payment opotions
SUBSCRIPTION newsletter_subscription   I have got to cancel the newsletter subscription
ACCOUNT    switch_account            can ya help me to change to the gold account
REFUND     track_refund              I am waiting for a compensation of {{Currency Symbol}}{{Refund Amount}}


In [3]:
categories = sorted(set(raw_dataset["category"]))
intents = sorted(set(raw_dataset["intent"]))
print(f"Категорій: {len(categories)}, намірів: {len(intents)}")
print(categories)

Категорій: 11, намірів: 27
['ACCOUNT', 'CANCEL', 'CONTACT', 'DELIVERY', 'FEEDBACK', 'INVOICE', 'ORDER', 'PAYMENT', 'REFUND', 'SHIPPING', 'SUBSCRIPTION']


### Дедуплікація

Частина звернень у датасеті повторюється дослівно. Якщо такий дублікат потрапить і в тренувальний, і у валідаційний набір,
validation loss буде заниженим: модель перевірятимуть на прикладах, які вона вже бачила.
Тому перед розбиттям залишаємо по одному прикладу для кожного тексту звернення (без урахування регістру й пробілів на краях).

In [4]:
seen, keep = set(), []
for i, text in enumerate(raw_dataset["instruction"]):
    key = text.strip().lower()
    if key not in seen:
        seen.add(key)
        keep.append(i)

dedup_dataset = raw_dataset.select(keep)
print(f"До дедуплікації: {len(raw_dataset)}, після: {len(dedup_dataset)}")

До дедуплікації: 26872, після: 24371


### Формат `messages`

Кожне звернення перетворюємо на діалог із трьох повідомлень:
- `system` — коротка інструкція;
- `user` — повідомлення клієнта;
- `assistant` — правильна відповідь: JSON із мітками.

Колонка `chat_template_kwargs` вимикає режим міркувань Qwen3: модель має одразу відповідати JSON.

In [5]:
SHORT_PROMPT = "Classify the customer message. Reply with JSON only."


def to_messages(example):
    answer = json.dumps({"category": example["category"], "intent": example["intent"]})
    return {
        "messages": [
            {"role": "system", "content": SHORT_PROMPT},
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": answer},
        ],
        "chat_template_kwargs": {"enable_thinking": False},
    }

### Тренувальний, валідаційний і тестовий набори

- **Тренувальний** (2800 прикладів) — на ньому модель вчиться.
- **Валідаційний** (200) — на ньому під час тренування обчислюється validation loss, щоб помітити перенавчання.
- **Тестовий** (1000) — на ньому лише один раз, після тренування, рахуємо підсумкові метрики.
  Під час тренування й налаштування ці приклади не використовуються, тому оцінка на них найчесніша.

Після дедуплікації та перемішування тестовий набір беремо з прикладів, які не потрапили до перших 3000.

Тестовий набір більший за валідаційний, бо від його розміру залежить точність вимірювання метрик:
на 1000 прикладах одна помилка змінює точність на 0,1%, тоді як на 200 — на 0,5%.
Донавчені моделі помиляються рідко, і на малому тестовому наборі різниця між ними може бути непомітною.

In [6]:
shuffled = dedup_dataset.shuffle(seed=42)

dataset = shuffled.select(range(3000)).map(to_messages, remove_columns=shuffled.column_names)
dataset = dataset.train_test_split(test_size=200, seed=42)
train_dataset, val_dataset = dataset["train"], dataset["test"]
test_dataset = shuffled.select(range(3000, 4000)).map(to_messages, remove_columns=shuffled.column_names)

print(f"Тренувальний: {len(train_dataset)}, валідаційний: {len(val_dataset)}, тестовий: {len(test_dataset)}")
train_dataset[0]["messages"]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:  15%|█▌        | 463/3000 [00:00<00:00, 2818.95 examples/s]

Map:  67%|██████▋   | 2000/3000 [00:00<00:00, 8604.82 examples/s]

Map: 100%|██████████| 3000/3000 [00:00<00:00, 8846.42 examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map: 100%|██████████| 1000/1000 [00:00<00:00, 12296.41 examples/s]

Тренувальний: 2800, валідаційний: 200, тестовий: 1000


[{'role': 'system',
  'content': 'Classify the customer message. Reply with JSON only.'},
 {'role': 'user',
  'content': 'I have got to locate hte bills from {{Person Name}}'},
 {'role': 'assistant',
  'content': '{"category": "INVOICE", "intent": "check_invoice"}'}]

## 2. Функції для генерації й оцінювання

Порівнюємо три моделі однієї сімʼї з однаковою архітектурою й даними попереднього тренування — **Qwen3** на 0,6, 1,7 і 4 млрд параметрів.
Усі вони мають той самий токенізатор і шаблон розмови, тому токенізатор завантажуємо один раз.

In [7]:
MODELS = ["Qwen/Qwen3-0.6B", "Qwen/Qwen3-1.7B", "Qwen/Qwen3-4B"]

tokenizer = AutoTokenizer.from_pretrained(MODELS[0])

# Сумісність із trl >= 0.20 (де assistant_only_loss=True вимагає тегів {% generation %} у шаблоні Qwen3)
if "{% generation %}" not in tokenizer.chat_template:
    _old_b = "{{- '<|im_start|>' + message.role + '\\n<think>\\n' + reasoning_content.strip('\\n') + '\\n</think>\\n\\n' + content.lstrip('\\n') }}"
    _new_b = "{{- '<|im_start|>' + message.role + '\\n<think>\\n' + reasoning_content.strip('\\n') + '\\n</think>\\n\\n' }}{%- generation -%}{{- content.lstrip('\\n') }}{%- endgeneration -%}"
    _old_e = "{{- '<|im_end|>\\n' }}\n    {%- elif message.role == \"tool\" %}"
    _new_e = "{%- generation -%}{{- '<|im_end|>\\n' }}{%- endgeneration -%}\n    {%- elif message.role == \"tool\" %}"
    if _old_b in tokenizer.chat_template and _old_e in tokenizer.chat_template:
        tokenizer.chat_template = tokenizer.chat_template.replace(_old_b, _new_b, 1).replace(_old_e, _new_e, 1)


### Детальний промпт

Без донавчання модель не знає переліку міток, тому для оцінки «до» передаємо його в промпті разом з описом формату.

In [8]:
LONG_PROMPT = f"""Classify the customer message for the support ticket routing system.
Reply with a single JSON object and nothing else, in the format:
{{"category": "<CATEGORY>", "intent": "<intent>"}}

Allowed categories: {", ".join(categories)}
Allowed intents: {", ".join(intents)}"""

for name, prompt in [("Короткий промпт", SHORT_PROMPT), ("Детальний промпт", LONG_PROMPT)]:
    print(f"{name}: {len(tokenizer(prompt)['input_ids'])} токенів")

Короткий промпт: 11 токенів
Детальний промпт: 166 токенів


### Метрики

Відповідь моделі вважаємо правильною, якщо програма може розібрати її як JSON і мітки збігаються з еталонними.
До донавчання моделі Qwen3 часто загортають JSON у блок розмітки Markdown (` ```json … ``` `), тому таку обгортку перед розбором знімаємо,
але окремо рахуємо, як часто відповідь є чистим JSON без зайвого тексту.

Рахуємо чотири метрики на всьому тестовому наборі:
- частку відповідей, які є чистим JSON;
- частку відповідей, які після зняття обгортки є валідним JSON з полями `category` і `intent`;
- точність категорії;
- точність наміру.

Щоб оцінка 1000 прикладів не тривала надто довго, відповіді генеруються батчами по 32 звернення.

In [9]:
def classify_batch(model, messages_batch, system_prompt, max_new_tokens=60):
    """Повертає відповіді моделі на список звернень клієнтів, згенеровані одним батчем."""
    prompts = [
        tokenizer.apply_chat_template(
            [{"role": "system", "content": system_prompt}, {"role": "user", "content": message}],
            tokenize=False, add_generation_prompt=True, enable_thinking=False,
        )
        for message in messages_batch
    ]
    # Для генерації батчем короткі промпти доповнюються зліва, щоб відповіді всіх промптів починалися в одній позиції
    tokenizer.padding_side = "left"
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    tokenizer.padding_side = "right"
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id
        )
    new_tokens = outputs[:, inputs["input_ids"].shape[1]:]
    return [answer.strip() for answer in tokenizer.batch_decode(new_tokens, skip_special_tokens=True)]


def classify(model, message, system_prompt):
    """Повертає відповідь моделі на одне звернення клієнта."""
    return classify_batch(model, [message], system_prompt)[0]


def is_clean_json(answer):
    """Чи є відповідь JSON без жодного зайвого тексту."""
    try:
        json.loads(answer)
        return True
    except json.JSONDecodeError:
        return False


def parse(answer):
    """Знімає обгортку ```json … ```, розбирає JSON; повертає None, якщо формат неправильний."""
    text = answer.strip()
    if text.startswith("```"):
        text = text.strip("`").removeprefix("json").strip()
    try:
        result = json.loads(text)
    except json.JSONDecodeError:
        return None
    if isinstance(result, dict) and {"category", "intent"} <= result.keys():
        return result
    return None


def evaluate(model, system_prompt, batch_size=32):
    """Рахує метрики на всьому тестовому наборі."""
    model.eval()
    messages = [ex["messages"][1]["content"] for ex in test_dataset]
    targets = [json.loads(ex["messages"][2]["content"]) for ex in test_dataset]
    answers = []
    for start in range(0, len(messages), batch_size):
        answers += classify_batch(model, messages[start:start + batch_size], system_prompt)

    clean = valid = category_ok = intent_ok = 0
    for answer, target in zip(answers, targets):
        clean += is_clean_json(answer)
        parsed = parse(answer)
        if parsed is not None:
            valid += 1
            category_ok += parsed["category"] == target["category"]
            intent_ok += parsed["intent"] == target["intent"]
    n = len(answers)
    metrics = {
        "Чистий JSON, %": 100 * clean / n,
        "Валідний JSON, %": 100 * valid / n,
        "Точність category, %": 100 * category_ok / n,
        "Точність intent, %": 100 * intent_ok / n,
    }
    return metrics, list(zip(messages, answers))


def show(answers, k=5):
    """Друкує перші k пар «звернення — відповідь моделі»."""
    for message, answer in answers[:k]:
        print(f"📩 {message}\n🤖 {answer}\n" + "-" * 60)


def free_memory():
    """Звільняє памʼять GPU після видалення моделей."""
    gc.collect()
    torch.cuda.empty_cache()

## 3. Чому не повне донавчання

Під час повного донавчання з оптимізатором Adam у змішаній точності на кожен параметр потрібно приблизно 16 байт:

| Що зберігається | Байтів на параметр |
|---|---|
| ваги у fp16 | 2 |
| градієнти у fp16 | 2 |
| копія ваг у fp32: у fp16 малі оновлення ваг губляться через округлення | 4 |
| два стани Adam у fp32: усереднені градієнт і його квадрат, за якими Adam підбирає крок для кожного параметра | 8 |
| **разом** | **16** |

До цього додаються активації, які залежать від довжини послідовностей і розміру батча.

Для Qwen3-4B це близько 64 GB без активацій, тоді як T4 має 15 GB: повне донавчання такої моделі потребує GPU рівня A100 на 80 GB.
Для Qwen3-0.6B потрібно лише близько 10 GB, тому в розділі 6 виконаємо повне донавчання саме цієї моделі.

## 4. Налаштування LoRA і тренування

LoRA заморожує всі ваги моделі й додає до вибраних шарів дві невеликі матриці $A$ і $B$.
Тренуються лише вони, тому градієнти й стани оптимізатора потрібні тільки для них.

Параметри `LoraConfig`:
- `r` — ранг матриць адаптера: що більший, то більше параметрів і «свободи» має адаптер;
- `lora_alpha` — масштаб внеску адаптера (зазвичай `2 × r`);
- `lora_dropout` — dropout в адаптері для регуляризації;
- `target_modules="all-linear"` — адаптери додаються до всіх лінійних шарів (attention і MLP).

In [10]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

Параметри тренування (`SFTConfig`):
- `assistant_only_loss=True` — loss рахується лише на відповідях асистента, а не на репліках користувача й системному промпті:
  для решти токенів мітки замінюються значенням −100, і функція втрат їх пропускає;
- `max_steps=100` — кількість кроків тренування; за один крок модель бачить 8 прикладів, тобто за все тренування — 800 із 2800,
  менше третини епохи. Для вузької задачі цього достатньо: validation loss виходить на плато раніше;
- `learning_rate=2e-4` — для LoRA learning rate приблизно в 10 разів більший, ніж для повного донавчання;
- `eval_strategy="steps"` і `eval_steps=10` — кожні 10 кроків модель оцінюється на валідаційному наборі:
  якщо validation loss почне зростати, модель перенавчається;
- `fp16=True` — змішана точність (T4 не підтримує bf16);
- `gradient_checkpointing=True` — активації не зберігаються, а перераховуються під час зворотного проходу: повільніше, але потрібно менше памʼяті.

In [11]:
def make_config(output_dir, learning_rate=2e-4):
    return SFTConfig(
        output_dir=output_dir,
        max_length=256,                  # діалог із цього датасету після шаблону займає близько 100 токенів
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,   # ефективний розмір батча: 4 × 2 = 8 прикладів
        per_device_eval_batch_size=8,
        learning_rate=learning_rate,
        lr_scheduler_type="cosine",
        warmup_steps=5,
        max_steps=100,
        eval_strategy="steps",
        eval_steps=10,
        fp16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        assistant_only_loss=True,
        logging_steps=10,
        save_strategy="no",
        report_to="none",
    )


def train(model, output_dir, peft_config=peft_config, learning_rate=2e-4):
    """Тренує модель (з LoRA, якщо задано peft_config) і повертає тренер, тривалість, пікову памʼять GPU та validation loss."""
    torch.cuda.reset_peak_memory_stats()
    trainer = SFTTrainer(
        model=model,
        args=make_config(output_dir, learning_rate),
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        peft_config=peft_config,
    )
    # Для QLoRA бібліотека TRL переводить ваги адаптера в bf16, який T4 не підтримує.
    # Повертаємо їх у fp32: змішана точність fp16 потребує, щоб параметри, що тренуються, були у fp32
    if getattr(model, "is_loaded_in_4bit", False):
        for param in trainer.model.parameters():
            if param.requires_grad:
                param.data = param.data.float()
    if peft_config is not None:
        trainer.model.print_trainable_parameters()
    start = time.time()
    trainer.train()
    eval_losses = [h["eval_loss"] for h in trainer.state.log_history if "eval_loss" in h]
    stats = {
        "Тривалість, хв": (time.time() - start) / 60,
        "Пікова памʼять GPU, GB": torch.cuda.max_memory_allocated() / 1e9,
        "Validation loss": eval_losses[-1],
    }
    return trainer, stats

## 5. LoRA на моделях різного розміру

Для кожної моделі виконуємо однакові кроки:
1. завантажуємо модель у fp16 (ваги Qwen3 збережено в bfloat16, але T4 не підтримує його апаратно);
2. оцінюємо її **до донавчання** з детальним промптом, у якому перелічено всі мітки;
3. донавчаємо методом LoRA з однаковими параметрами;
4. оцінюємо **після донавчання** з коротким промптом без переліку міток;
5. зберігаємо адаптер і вивантажуємо модель з GPU, щоб звільнити памʼять для наступної.

In [12]:
results = {}


def run_lora(model_name):
    """Повний цикл для однієї моделі: оцінка до, LoRA, оцінка після, збереження адаптера."""
    short_name = model_name.split("/")[-1]
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16, device_map="auto")
    print(f"{short_name}: {model.num_parameters() / 1e9:.2f} млрд параметрів, "
          f"памʼять GPU після завантаження: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

    before_metrics, before_answers = evaluate(model, LONG_PROMPT)
    print("\nДо донавчання:")
    show(before_answers, k=3)

    trainer, stats = train(model, f"{short_name}-lora")
    after_metrics, after_answers = evaluate(trainer.model, SHORT_PROMPT)
    print("\nПісля донавчання:")
    show(after_answers, k=3)

    adapter_dir = f"{short_name}-lora-adapter"
    trainer.save_model(adapter_dir)
    stats["Адаптер, MB"] = os.path.getsize(os.path.join(adapter_dir, "adapter_model.safetensors")) / 1e6

    results[short_name] = {"до": before_metrics, "LoRA": after_metrics | stats}
    del trainer, model
    free_memory()
    return pd.DataFrame(results[short_name]).round(2)

In [13]:
run_lora("Qwen/Qwen3-0.6B")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Qwen3-0.6B: 0.60 млрд параметрів, памʼять GPU після завантаження: 1.2 GB



До донавчання:
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "CONTACT", "intent": "contact_customer_service"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "CONTACT", "intent": "contact_customer_service"}
------------------------------------------------------------


Tokenizing train dataset:   0%|          | 0/2800 [00:00<?, ? examples/s]

Tokenizing train dataset:   9%|▉         | 247/2800 [00:00<00:01, 2452.84 examples/s]

Tokenizing train dataset:  19%|█▉        | 526/2800 [00:00<00:00, 2643.85 examples/s]

... (16 outputs left out)

trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650


Step,Training Loss,Validation Loss
10,1.363800,0.476443
20,0.320000,0.140350
30,0.077300,0.057441
40,0.056700,0.032348
50,0.024100,0.011602
60,0.006700,0.007891
70,0.013900,0.008914
80,0.009300,0.008024
90,0.006000,0.007947
100,0.019300,0.007799



Після донавчання:
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "FEEDBACK", "intent": "review"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "ACCOUNT", "intent": "registration_problems"}
------------------------------------------------------------


,до,LoRA
"Чистий JSON, %",100.0,100.00
"Валідний JSON, %",100.0,100.00
"Точність category, %",37.9,99.70
"Точність intent, %",25.6,97.50
"Тривалість, хв",NaN,2.59
"Пікова памʼять GPU, GB",NaN,2.16
Validation loss,NaN,0.01
"Адаптер, MB",NaN,40.42


In [ ]:
run_lora("Qwen/Qwen3-1.7B")

In [ ]:
run_lora("Qwen/Qwen3-4B")

### Порівняння

Чи може менша модель після донавчання досягти якості більшої? Порівняйте точність, тривалість тренування, памʼять і розмір адаптера.

In [14]:
pd.DataFrame({
    f"{name} {stage}": metrics
    for name, stages in results.items()
    for stage, metrics in stages.items()
}).round(2)

,Qwen3-0.6B до,Qwen3-0.6B LoRA
"Чистий JSON, %",100.0,100.00
"Валідний JSON, %",100.0,100.00
"Точність category, %",37.9,99.70
"Точність intent, %",25.6,97.50
"Тривалість, хв",NaN,2.59
"Пікова памʼять GPU, GB",NaN,2.16
Validation loss,NaN,0.01
"Адаптер, MB",NaN,40.42


## 6. Повне донавчання Qwen3-0.6B

Модель на 0,6 млрд параметрів потребує для повного донавчання близько 10 GB, тому на T4 її можна донавчити повністю й порівняти з LoRA.

Відмінності від LoRA:
- `peft_config=None` — тренуються всі ваги моделі;
- модель завантажуємо у **fp32**: змішана точність fp16 тримає головну копію ваг у fp32, а в fp16 лише обчислює прохід;
- `learning_rate=2e-5` — для повного донавчання learning rate приблизно в 10 разів менший, ніж для LoRA.

In [ ]:
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B", dtype=torch.float32, device_map="auto")
trainer, full_stats = train(model, "Qwen3-0.6B-full", peft_config=None, learning_rate=2e-5)
full_metrics, full_answers = evaluate(trainer.model, SHORT_PROMPT)
show(full_answers, k=3)

del trainer, model
free_memory()

pd.DataFrame({"Qwen3-0.6B LoRA": results["Qwen3-0.6B"]["LoRA"], "Qwen3-0.6B повне": full_metrics | full_stats}).round(2)

## 7. Збереження і завантаження адаптера

`save_model` зберігає **лише адаптер** (`adapter_model.safetensors`) разом із його налаштуваннями й токенізатором, а не всю модель.

In [ ]:
adapter_dir = "Qwen3-4B-lora-adapter"
for name in sorted(os.listdir(adapter_dir)):
    size = os.path.getsize(os.path.join(adapter_dir, name))
    print(f"{name:<35} {size / 1e6:>8.1f} MB")
print("\nДля порівняння, ваги Qwen3-4B у fp16: ~8 GB")

Щоб використати адаптер, завантажуємо модель і накладаємо на неї адаптер з диска.
Метод `merge_and_unload` вливає адаптер у ваги моделі: $W' = W + \frac{\alpha}{r} BA$.
Після цього модель працює як звичайна, без додаткових обчислень для адаптера.

In [ ]:
from peft import PeftModel

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-4B", dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(model, adapter_dir)
model = model.merge_and_unload()
print(type(model).__name__)

# Злиту модель можна зберегти як звичайну: model.save_pretrained("qwen3-4b-merged")

Перевіримо злиту модель на власних повідомленнях:

In [ ]:
my_messages = [
    "Hi, I ordered a laptop last week and it still hasn't arrived. Where is it?",
    "I was charged twice for the same order, please help",
    "How can I talk to a real person?",
    "I guess, you have shipped to a wrong address",
]
for message in my_messages:
    print(f"📩 {message}\n🤖 {classify(model, message, SHORT_PROMPT)}\n")

In [ ]:
del model
free_memory()

## 8. QLoRA

QLoRA завантажує заморожені ваги Qwen3-4B в **4 бітах** (формат NF4), а адаптери LoRA тренує у звичайній точності.
Код тренування той самий, змінюється лише завантаження моделі.

### Формат NF4

У 4 бітах вага може набувати лише одного з 16 значень. Ваги ділять на блоки по 64, і для кожного блоку
зберігають масштаб — найбільше за модулем значення в блоці. Формати відрізняються тим, як розташовані ці 16 значень:
- **INT4** — 16 рівновіддалених значень. Ваги нейромережі розподілені приблизно нормально: більшість близька до нуля,
  а великих за модулем мало. Тому крайні значення використовуються рідко, а біля нуля їх бракує;
- **NF4 (NormalFloat4)** — 16 значень розташовані за квантилями нормального розподілу: густо біля нуля й рідко на краях.
  Кожне значення використовується приблизно однаково часто, тому помилка квантизації менша, ніж у INT4.

NF4 — формат лише для зберігання: перед кожним множенням ваги деквантизуються у fp16 (`bnb_4bit_compute_dtype`).
Детальніше — у блозі Hugging Face [Making LLMs even more accessible with bitsandbytes, 4-bit quantization and QLoRA](https://huggingface.co/blog/4bit-transformers-bitsandbytes)
і в статті [QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314).

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",              # 4-бітний формат, розрахований на нормально розподілені ваги
    bnb_4bit_compute_dtype=torch.float16,   # обчислення виконуються у fp16
    bnb_4bit_use_double_quant=True,         # квантизуються також масштаби квантизації: ще трохи менше памʼяті
)

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-4B", quantization_config=bnb_config, device_map="auto")
print(f"Памʼять GPU після завантаження: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
trainer, qlora_stats = train(model, "Qwen3-4B-qlora")
qlora_metrics, _ = evaluate(trainer.model, SHORT_PROMPT)
qlora_stats

## 9. Підсумкове порівняння

In [ ]:
summary = {
    f"{name} {stage}": metrics
    for name, stages in results.items()
    for stage, metrics in stages.items()
}
summary["Qwen3-0.6B повне"] = full_metrics | full_stats
summary["Qwen3-4B QLoRA"] = qlora_metrics | qlora_stats
pd.DataFrame(summary).round(2)

**Що варто помітити:**
- LoRA тренує близько 1–2% параметрів моделі, а адаптер займає десятки або сотні мегабайт замість гігабайтів для всієї моделі.
- Порівняйте моделі різного розміру: наскільки менша модель після донавчання поступається більшій і скільки вона заощаджує часу й памʼяті.
  Для вузької задачі з тисячами прикладів донавчена мала модель може бути вигіднішою за велику.
- Порівняйте повне донавчання Qwen3-0.6B з LoRA на тій самій моделі за якістю, часом і памʼяттю.
- До донавчання моделі отримують детальний промпт із переліком міток, після — лише короткий:
  формат і значення міток тепер зберігаються у вагах, а не в промпті. Коротший промпт означає швидшу й дешевшу обробку кожного запиту.
- QLoRA потребує помітно менше памʼяті, ніж LoRA на тій самій моделі, але тренується повільніше, бо ваги щоразу деквантизуються під час обчислень.

## 0.1. Теоретичні та практичні питання (22–30)

### 22. Скільки часу зайняло виконання `03_sft_lora`? Які проблеми виникли під час виконання?
- **Час виконання на локальній GPU (`NVIDIA GeForce GTX 1070`, 8 ГБ VRAM):** підготовка даних `Bitext`, оцінка `Qwen/Qwen3-0.6B` до/після, LoRA-навчання (100 кроків) та перевірка злиття адаптера зайняли разом **`346.2 с` (~`5.77 хв`)**, із яких саме LoRA-тренування `Qwen3-0.6B` тривало **`2.59 хв`**, а повний цикл `run_lora("Qwen/Qwen3-0.6B")` (включно з двома прогонами по 1 000 тестів) — `324.3 с`.
- **Проблеми та виявлені нюанси під час виконання:**
  1. **Несумісність `chat_template` `Qwen3` із `trl 0.29.1` при `assistant_only_loss=True`:** бібліотека `trl >= 0.20` викликає `return_assistant_tokens_mask=True`, яка кидає `RuntimeError`, якщо у шаблоні розмови немає Jinja-тегів `{% generation %}`. У клітинці 13 було додано 6-рядковий сумісний патч до `tokenizer.chat_template`, після чого маскування спрацювало коректно (`Validation loss` впав до `0.01`, а точність `intent` піднялася з **`25.6%` до `97.50%`**).
  2. **Обмеження локальних 8 ГБ VRAM проти Colab T4 (15 ГБ):**
     - `Qwen3-0.6B` (LoRA `fp16`) умістився легко (**`1.2 GB`** після завантаження, пікова **`2.16 GB`**).
     - Натомість `Qwen3-4B` у `fp16` (`~8.04 GB` лише для ваг), повне донавчання `Qwen3-0.6B` у `float32` (`~9.6+ GB` з оптимізатором Adam) та `Qwen3-4B` **QLoRA** (потрібен пакет `bitsandbytes` 4-бітної квантизації) не запускалися в межах локальної 8 ГБ Windows-конфігурації (тому в клітинці 46 злиття адаптера `merge_and_unload()` додатково перевірено саме на `Qwen3-0.6B-lora-adapter`).

Нижче наведено живу локальну перевірку збереженого адаптера `Qwen3-0.6B-lora-adapter` (`40.42 MB`), `merge_and_unload()` та інференсу на власних зверненнях:

In [15]:
# Локальна перевірка злиття адаптера Qwen3-0.6B-lora-adapter (аналог розділу 7 на 8 GB GPU)
from peft import PeftModel

_ad_dir = "Qwen3-0.6B-lora-adapter"
for _fn in sorted(os.listdir(_ad_dir)):
    _sz = os.path.getsize(os.path.join(_ad_dir, _fn))
    print(f"{_fn:<35} {_sz / 1e6:>8.2f} MB")

_m = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B", dtype=torch.float16, device_map="auto")
_m = PeftModel.from_pretrained(_m, _ad_dir).merge_and_unload()
print("Merged model class:", type(_m).__name__)

_test_msgs = [
    "Hi, I ordered a laptop last week and it still hasn't arrived. Where is it?",
    "I was charged twice for the same order, please help",
    "How can I talk to a real person?",
    "I guess, you have shipped to a wrong address",
]
for _msg in _test_msgs:
    print(f"📩 {_msg}\n🤖 {classify(_m, _msg, SHORT_PROMPT)}\n")

del _m
free_memory()

README.md                               0.01 MB
adapter_config.json                     0.00 MB
adapter_model.safetensors              40.42 MB
added_tokens.json                       0.00 MB
chat_template.jinja                     0.00 MB
merges.txt                              1.67 MB
special_tokens_map.json                 0.00 MB
tokenizer.json                         11.42 MB
tokenizer_config.json                   0.01 MB
training_args.bin                       0.01 MB
vocab.json                              2.78 MB


Merged model class: Qwen3ForCausalLM


📩 Hi, I ordered a laptop last week and it still hasn't arrived. Where is it?
🤖 {"category": "DELIVERY", "intent": "delivery_issue"}



📩 I was charged twice for the same order, please help
🤖 {"category": "ORDER", "intent": "cancel_order"}



📩 How can I talk to a real person?
🤖 {"category": "CONTACT", "intent": "contact_human_agent"}



📩 I guess, you have shipped to a wrong address
🤖 {"category": "SHIPPING", "intent": "set_up_shipping_address"}



---

### 23. Прокоментуйте рядок «Памʼять GPU після завантаження: …» — чи приблизно стільки очікувалося, поясніть чому?
У запуску для **`Qwen/Qwen3-0.6B`** отримано рядок:
`Qwen3-0.6B: 0.60 млрд параметрів, памʼять GPU після завантаження: 1.2 GB`

- **Чи стільки очікувалося?** Так, практично байт у байт!
- **Пояснення за формулою з лекції:** кожен параметр моделі у форматі `fp16` (`torch.float16`, 16 біт) займає `2` байти, тому лише ваги моделі в пам'яті GPU оцінюються як:
  $$M_{\text{weights}} \approx P \times 2 \text{ байт}$$
  Для `0.60 млрд` параметрів це $0.60 \times 2 = \mathbf{1.20\text{ GB}}$.
- У момент одразу після `.from_pretrained(..., dtype=torch.float16)` у пам'яті GPU ще немає ні великих проміжних активацій (окрім буферів RoPE/вирівнювання PyTorch), ні градієнтів, ні станів Adam — тому `torch.cuda.memory_allocated()` дорівнює масі самих ваг моделі. (Для `Qwen3-1.7B` і `Qwen3-4B` у `fp16` це аналогічно ~`3.4 GB` та ~`8.0 GB`).

---

### 24. Що робить конструкція `with torch.no_grad():` у функції `classify()`?
- Тимчасово вимикає двигун автодиференціювання (*autograd*) всередині контекстного блоку: PyTorch **не будує обчислювальний граф** і не зберігає активації кожного шару для подальшого `.backward()`.
- Під час інференсу (класифікації) градієнти не потрібні, тому `with torch.no_grad():` дає подвійну вигоду:
  1. **Різко зменшує витрату VRAM** (не накопичуються проміжні тензори при авторегресійній генерації до 60 токенів батчами по 32 приклади).
  2. **Пришвидшує обчислення**, прибираючи службову логіку трекінгу графа.

---

### 25. Чому у функції `classify()` вказано `do_sample=False`?
- `do_sample=False` вимикає випадковий вибір токенів (семплювання за температурою/`top_p`/`top_k`) і перемикає генерацію в **детерміноване жадібне декодування (*greedy decoding*)** — на кожному кроці модель завжди бере найімовірніший токен ($\arg\max$).
- Для **класифікації в строгий JSON** креативність шкідлива: потрібні **стабільність, відсутність випадкових описок у назвах міток** і 100% відтворюваність оцінки.

---

### 26. Чому підхід зі зміною голови, як у демо 1, не підходить для цього завдання?
1. **Ієрархічний структурований JSON замість плоского класу:** у демо 1 лінійна голова `Linear(768, 2)` повертає лише один номер класу. Тут модель має генерувати валідний **JSON із двома пов'язаними полями** (`{"category": ..., "intent": ...}` — 11 категорій і 27 намірів).
2. **Розширюваність та збереження генеративної природи:** генеративну модель після SFT + LoRA легко навчити повертати додаткові поля (`priority`, `language`, `order_id`), нові категорії чи працювати у звичайному чат-режимі — без перероблення голови з нуля, і навіть перемикати кілька LoRA-адаптерів на одному базовому скелеті `AutoModelForCausalLM`.
3. **Архітектура decoder-only:** `Qwen3` — каузальна генеративна модель без енкодерного токена `[CLS]`; її природний інтерфейс — авторегресійна генерація токенів відповіді.

---

### 27. Скільки токенів займає короткий та детальний промпти у розділі 2? Як це вплине на витрати токенів при обробці 10 000 запитів на день? Порахуйте для обох варіантів.
За реальним виміром токенізатором `Qwen/Qwen3-0.6B` у клітинці 15:
- **Короткий промпт (`SHORT_PROMPT`):** **`11` токенів**.
- **Детальний промпт (`LONG_PROMPT`):** **`166` токенів** *(у **15.1× більше**, бо доводиться перераховувати всі 11 категорій та 27 намірів і правила JSON)*.

**Розрахунок витрат лише на системний промпт для `10 000` запитів/день:**
- **Із детальним промптом (без донавчання):**
  - За день: $166 \times 10\,000 = \mathbf{1\,660\,000}$ токенів/день (`1.66M` вхідних токенів).
  - За 30 днів: $1\,660\,000 \times 30 = \mathbf{49\,800\,000}$ токенів/міс (`49.8M`).
- **Із коротким промптом (після LoRA SFT):**
  - За день: $11 \times 10\,000 = \mathbf{110\,000}$ токенів/день (`0.11M` вхідних токенів).
  - За 30 днів: $110\,000 \times 30 = \mathbf{3\,300\,000}$ токенів/міс (`3.3M`).
- **Економія:** SFT-адаптер «зашиває» всю схему міток у ваги моделі й економить **`1 550 000` токенів щодня** (**`46.5 млн` токенів щомісяця**, тобто скорочує витрати на системний промпт на **93.4%**), при цьому точність `intent` на `Qwen3-0.6B` ще й зростає з **`25.6%` до `97.50%`**!

---

### 28. Навіщо в датасеті передається `chat_template_kwargs={"enable_thinking": False}`? Що сталося б, якби ми донавчили модель в режимі `/think`?
- **`enable_thinking=False`** вказує `apply_chat_template` вставляти порожній маркер `<think>\n\n</think>\n\n` перед відповіддю асистента, щоб `Qwen3` вчилася **одразу генерувати чистий JSON** без проміжних роздумів.
- **Що сталося б при донавчанні в режимі `/think`:**
  1. **Без роздумів у датасеті:** шаблон вимагав би міркувань, а еталон містив би лише JSON — модель почала б плутати чи ламати режим `/think` після SFT.
  2. **Із доданими міркуваннями `<think>…</think>` перед JSON:** модель генерувала б по `100–300+` токенів тексту на кожен запит (у багато разів повільніше та дорожче на інференсі), а при поточному ліміті `max_new_tokens=60` генерація **обривалася б усередині `<think>…</think>`**, не встигаючи видати сам JSON (`Валідний JSON` впав би майже до `0%`).

---

### 29. Що означає параметр `assistant_only_loss=True` в `SFTConfig`? Які токени будуть навчатися, якщо його вимкнути? Чому це погано для цієї задачі?
- **`assistant_only_loss=True`** позначає всі службові та користувацькі токени маскою `-100` і змушує `SFTTrainer` рахувати функцію втрат **лише на токенах відповіді асистента** (сам JSON `{"category": ..., "intent": ...}` та токен зупинки `<|im_end|>`).
- **Якщо вимкнути (`assistant_only_loss=False`):** втрати обчислюватимуться на **всій послідовності цілком** — модель вчитиметься передбачати текст однакового `SHORT_PROMPT` (`system`) і довільні звернення користувачів (`user`).
- **Чому це погано тут:** звернення клієнтів містять багато варіативного тексту та заповнювачів (`{{Order Number}}`), і більшість токенів у прикладі — саме `system` + `user`. Якщо штрафувати модель за помилки у тексті `user`, оптимізація розпорошується на запам'ятовування вхідних фраз замість точного мапінгу «звернення → JSON-класифікація», що погіршує узагальнення.

---

### 30. Порівняйте результати LoRA та QLoRA по пам'яті, часу тренування та якості (розділ 9). Коли варто обрати QLoRA?
| Критерій | LoRA (`fp16`) | QLoRA (`NF4 4-bit + double quant`) |
| --- | --- | --- |
| **Пам'ять GPU** | Базові ваги у 16 бітах (`~8.0 GB` для 4B; пік ~10–12 GB) | Базові ваги стискаються у ~3.5–4× (`~2.5–3.2 GB` для 4B; пік ~4.5–6 GB) — економія понад 50% VRAM |
| **Час тренування** | **Швидша** (матричні множення безпосередньо у `fp16`) | **Повільніша** (~на 20–40%), оскільки перед кожним лінійним шаром 4-бітні ваги `NF4` доводиться деквантизувати в `fp16` |
| **Якість** | Висока еталонна якість LoRA | Практично **на рівні LoRA** (різниця зазвичай < 0.5–1% завдяки нерівномірним квантилям `NF4`, які щільніше розміщені біля нуля, та обчисленням адаптера у `fp16/fp32`) |

**Коли варто обрати QLoRA:** тоді, коли сильніша базова модель у `fp16` **не вміщається у доступну VRAM вашої GPU** (наприклад, 4B–7B модель на 8 ГБ карті або 7B–14B на 16 ГБ T4) і ви готові пожертвувати частиною швидкості тренування в обмін на доступ до більшої моделі без `OutOfMemoryError`.